In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.integrate import cumulative_trapezoid
from scipy.integrate import trapezoid


'''
Linearized Schwarzchild metric with Single BH at the center

Code is based on the original working code. Evaluate z using cumulative trapezoidal integration.
'''

# =========================
# metric
# =========================
'''ADD THE CONDITIONAL GRAVITATIONAL POTENTIAL'''
def g_xx(x, y):
    r = np.sqrt(x**2 + y**2)
    return 1 + 2*GM/r
def g_yy(x, y):
    r = np.sqrt(x**2 + y**2)
    return 1 + 2*GM/r
def g_xy(x, y):
    return None

# =========================
# embedding pde's
# =========================
def partial_x_Z(x, y):
    sign = pick_partialx_sign(x, y)
    return sign * np.sqrt(g_xx(x, y) - 1)
def partial_y_Z(x, y):
    sign = pick_partialx_sign(x, y)
    return sign * np.sqrt(g_yy(x, y) - 1)
# sign checker
def sign_condition_check(x, y, sign_parital_x, sign_parital_y):
    condition_met = False
    dz_dr = x * sign_parital_x*partial_x_Z(x, y) + y * sign_parital_y*partial_y_Z(x, y)
    if dz_dr > 0:
        condition_met = True
    return condition_met
def pick_partialx_sign(x_coord, y_coord):
    '''
    picks either the positive or negative solution at each point in space
    '''
    partialx_sign, partialy_sign = 1, 1
    if sign_condition_check(x_coord, y_coord, partialx_sign, partialy_sign):
        return partialx_sign
    else:
        partialx_sign, partialy_sign = -1, 1
        if sign_condition_check(x_coord, y_coord, partialx_sign, partialy_sign):
            return partialx_sign
        else:
            partialx_sign, partialy_sign = 1, -1
            if sign_condition_check(x_coord, y_coord, partialx_sign, partialy_sign):
                return partialx_sign
            else:
                partialx_sign, partialy_sign = -1, -1
                if sign_condition_check(x_coord, y_coord, partialx_sign, partialy_sign):
                    return partialx_sign
                else:
                    print(f'\nERROR\npick_partialx_sign() : NO SIGN PAIR FOUND AT ({x_coord}, {y_coord})\n')
    return None


# =========================
# Set-up Values
# =========================
# constants (change if need)
GM = 1
x_start, x_end = -100, 100
y_start, y_end = -100, 100
n_steps = 100
# coordinate grids
x_axis = np.linspace(x_start, x_end, n_steps)
y_axis = np.linspace(y_start, y_end, n_steps)
x_meshed, y_meshed = np.meshgrid(x_axis, y_axis)
r = np.sqrt(x_meshed**2 + y_meshed**2)
# creating a 2d np array to copy later on
grid = np.zeros((y_axis.size, x_axis.size))
# more constants
dx, dy = (x_end - x_start) / (n_steps-1), (y_end - y_start) / (n_steps-1)
boundary_condition = 2*np.sqrt(r[:,0:1] - 1)
boundary_condition = boundary_condition * np.ones_like(grid)

# =========================
# Calculation of Z
# =========================
# calculating a(x,y) and b(x,y)
a = np.where(r<=1.1*GM, 0, partial_x_Z(x_meshed, y_meshed))
b = np.where(r<=1.1*GM, 0, partial_y_Z(x_meshed, y_meshed))
# calculating A: cumulative integral
A = cumulative_trapezoid(a, x=x_axis, axis=1, initial=0)
# calculating Z
Z = A + boundary_condition

# ===================================
#  checking c(y)'s x-independence
# ===================================
# calculating partial A
partial_y_A = np.gradient(A, y_axis, axis=0)
#calculating d/dy[c(y)]
dc_dy = b - partial_y_A
# calculating c(y) to check its x-independence: cumulative integral
c_check = cumulative_trapezoid(dc_dy, x=y_axis, axis=0, initial=0)

### plotting
# the function
def plot_3d_and_slice(surface, surface_name):
    num_slices = 4

    fig= plt.figure(figsize=(10, 6))
    ax_3d = fig.add_subplot(221, projection='3d')
    ax_slicex = fig.add_subplot(223)
    ax_slicey = fig.add_subplot(224)

    # from matplotlib example (modified)
    surf = ax_3d.plot_surface(x_meshed, y_meshed, surface, cmap=cm.coolwarm, linewidth=0, antialiased=False)
    ax_3d.zaxis.set_major_formatter('{x:.02f}')
    fig.colorbar(surf, shrink=0.5, aspect=5)
    ax_3d.set(xlabel='x', ylabel='y', title=surface_name)
    # ax_3d.set_zlim(bottom=0)

    color = ['b', 'g', 'y' , 'r']
    while len(color) < num_slices:
        color.append['r']
    
    for i in range(num_slices):
         ax_slicex.plot(x_axis, surface[int(len(surface)*(1/2 + i/2/num_slices))], color[i], label=f'y = {int(100*i/2/num_slices)} r_s')
         ax_slicex.set(xlabel='x', ylabel=surface_name, title=f'{surface_name} vs x')
         ax_slicey.plot(y_axis, surface[:,int(len(surface)*(1/2 + i/2/num_slices))], color[i], label=f'x = {int(100*i/2/num_slices)} r_s')
         ax_slicey.set(xlabel='y', ylabel=surface_name, title=f'{surface_name} vs y')
    ax_slicex.legend()
    ax_slicey.legend()
    
    plt.tight_layout
    plt.show()
    return None
# a
plot_3d_and_slice(a, 'a')
# b
plot_3d_and_slice(b, 'b')
# A
# plot_3d_and_slice(A, 'A')
# boundary condition
plot_3d_and_slice(boundary_condition, 'boundary condition')
# Z
plot_3d_and_slice(Z, 'Z')
# c(y)
plot_3d_and_slice(c_check, 'c(y)')
'''
# partial_y_A
plot_3d_and_slice(partial_y_A, 'A_y')
# dc_dy
plot_3d_and_slice(dc_dy, 'dc/dy')
'''

RecursionError: maximum recursion depth exceeded